In [1]:
!pip install librosa lightgbm optuna pyloudnorm --quiet

In [2]:
import os
import subprocess
import warnings
import kagglehub
import pickle
import numpy as np
import librosa
import pyloudnorm as pyln
from scipy.stats import kurtosis, skew

warnings.filterwarnings('ignore')

In [3]:

SAMPLE_RATE     = 16000
CLIP_DURATION   = 3
TARGET_LEN      = SAMPLE_RATE * CLIP_DURATION 
N_MFCC          = 13
N_MELS          = 40
N_MFCC_QUARTERS = 4
TARGET_LUFS     = -23.0   
CONFIDENCE_THR  = 0.75  

In [4]:
path = kagglehub.dataset_download("shafayatulislam/real-audio-data")
print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/shafayatulislam/real-audio-data


In [5]:
ORIG_AUDIO_PATH = "/kaggle/input/datasets/shafayatulislam/real-audio-data/16 Apr 11.07pm_.m4a"
WAV_AUDIO_PATH  = "/kaggle/working/test_audio.wav"

result = subprocess.run(
    ["ffmpeg", "-y", "-i", ORIG_AUDIO_PATH,
     "-ar", str(SAMPLE_RATE),  
     "-ac", "1",             
     WAV_AUDIO_PATH],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("conversion failed")
    print(result.stderr[-500:])  
    TEST_AUDIO_PATH = ORIG_AUDIO_PATH
else:
    TEST_AUDIO_PATH = WAV_AUDIO_PATH

In [6]:
test_audio, _ = librosa.load(
    TEST_AUDIO_PATH,
    sr=SAMPLE_RATE,
    duration=CLIP_DURATION,
    mono=True
)

if len(test_audio) < TARGET_LEN:
    test_audio = np.pad(test_audio, (0, TARGET_LEN - len(test_audio)))

print(f"Loaded  : {len(test_audio)} samples  ({len(test_audio)/SAMPLE_RATE:.2f}s)")

Loaded  : 48000 samples  (3.00s)


In [7]:
rms_raw = float(np.sqrt(np.mean(test_audio ** 2)))
print(f"Waveform stats (raw)")
print(f"  samples : {len(test_audio)}")
print(f"  min     : {test_audio.min():.4f}")
print(f"  max     : {test_audio.max():.4f}")
print(f"  RMS     : {rms_raw:.4f}")

if rms_raw < 1e-4:
    print("\nThe first 3 seconds appear nearly silent")
else:
    print("\nAudio level looks good")

Waveform stats (raw)
  samples : 48000
  min     : -0.3481
  max     : 0.4584
  RMS     : 0.0314

Audio level looks good


In [8]:
def normalize_loudness(audio: np.ndarray, sr: int = SAMPLE_RATE,
                       target_lufs: float = TARGET_LUFS) -> np.ndarray:
    meter = pyln.Meter(sr)
    audio64 = audio.astype(np.float64)
    loudness = meter.integrated_loudness(audio64)
    if not (np.isinf(loudness) or np.isnan(loudness)):
        audio64 = pyln.normalize.loudness(audio64, loudness, target_lufs)
    else:
        print("Loudness measurement returned inf/nan")
    return np.clip(audio64, -1.0, 1.0).astype(np.float32)

test_audio = normalize_loudness(test_audio)

In [9]:
# Verify
rms_norm = float(np.sqrt(np.mean(test_audio ** 2)))
print(f"Waveform stats (after loudness normalization)")
print(f"  RMS     : {rms_norm:.4f}")
print(f"  min     : {test_audio.min():.4f}")
print(f"  max     : {test_audio.max():.4f}")
print("Loudness normalization applied (@", TARGET_LUFS, "LUFS)")

Waveform stats (after loudness normalization)
  RMS     : 0.0488
  min     : -0.5401
  max     : 0.7114
Loudness normalization applied (@ -23.0 LUFS)


In [10]:
def extract_features(audio, sr=SAMPLE_RATE):
    feats = []
    mfcc    = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)
    d_mfcc  = librosa.feature.delta(mfcc)
    d2_mfcc = librosa.feature.delta(mfcc, order=2)

    n_frames = mfcc.shape[1]
    q_size   = max(1, n_frames // N_MFCC_QUARTERS)

    for matrix in (mfcc, d_mfcc, d2_mfcc):
        for q in range(N_MFCC_QUARTERS):
            seg = matrix[:, q * q_size : (q + 1) * q_size]
            if seg.shape[1] == 0:
                seg = matrix[:, -1:]
            feats += list(np.mean(seg, axis=1))
            feats += list(np.std(seg,  axis=1))

    feats += list(kurtosis(mfcc, axis=1, nan_policy='omit'))
    feats += list(skew(    mfcc, axis=1, nan_policy='omit'))

    mel    = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=N_MELS)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    feats += list(np.mean(mel_db, axis=1))
    feats += list(np.std( mel_db, axis=1))

    contrast = librosa.feature.spectral_contrast(y=audio, sr=sr, n_bands=6)
    feats += list(np.mean(contrast, axis=1))
    feats += list(np.std( contrast, axis=1))

    chroma = librosa.feature.chroma_stft(y=audio, sr=sr)
    feats += list(np.mean(chroma, axis=1))
    feats += list(np.std( chroma, axis=1))

    for feat_fn in (
        lambda: librosa.feature.zero_crossing_rate(y=audio),
        lambda: librosa.feature.spectral_centroid(y=audio, sr=sr),
        lambda: librosa.feature.spectral_rolloff(y=audio,  sr=sr),
        lambda: librosa.feature.spectral_bandwidth(y=audio, sr=sr),
        lambda: librosa.feature.rms(y=audio),
    ):
        v = feat_fn()
        feats += [float(np.mean(v)), float(np.std(v))]

    return np.array(feats, dtype=np.float32)

In [11]:
file_path1 = '/kaggle/input/datasets/shafayatulislam/trainedmodels/encoder_v3.pkl'
file_path2 = '/kaggle/input/datasets/shafayatulislam/trainedmodels/ensemble_v3.pkl'
file_path3 = '/kaggle/input/datasets/shafayatulislam/trainedmodels/scaler_v3.pkl'

In [12]:
with open(file_path3, 'rb') as f:
    scaler = pickle.load(f)
with open(file_path1, 'rb') as f:
    encoder = pickle.load(f)
with open(file_path2, 'rb') as f:
    ensemble = pickle.load(f)

In [13]:
raw_features        = extract_features(test_audio)
normalized_features = scaler.transform(raw_features.reshape(1, -1))[0]

print(f"Feature vector: {raw_features.shape[0]} dims")
print(f"Any NaN/Inf: {np.any(~np.isfinite(normalized_features))}")

Feature vector: 466 dims
Any NaN/Inf: False


In [14]:
probabilities   = ensemble.predict_proba(normalized_features.reshape(1, -1))[0]
max_index       = np.argmax(probabilities)
max_confidence  = probabilities[max_index]
predicted_label = encoder.classes_[max_index]

In [15]:
# Print the result
for i, class_name in enumerate(encoder.classes_):
    prob_percentage = probabilities[i] * 100
    print(f"{class_name.capitalize():<8} : {prob_percentage:>5.1f}%")

Crackle  :  11.1%
Normal   :   9.3%
Snore    :   9.4%
Wheeze   :  70.2%


In [16]:
if max_confidence >= CONFIDENCE_THR:
    print(f"[DETECTED]  {predicted_label.capitalize()}  ({max_confidence*100:.1f}%)")
else:
    print(f"[DISCARDED] Below {CONFIDENCE_THR*100:.0f}% threshold")
    print(f"\nHighest: {predicted_label.capitalize()} at {max_confidence*100:.1f}%")

[DISCARDED] Below 75% threshold

Highest: Wheeze at 70.2%
